In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualizations
%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

dataset_path = '/content/drive/MyDrive/Tech Scholarship Data Science Curriculum & Materials/notebooks-&-dataset/dataset'
BASE_URL = f"{dataset_path}/"
# Define files
files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv"
}

print("Downloading and caching Olist Datasets for auditing...")

try:
    orders = pd.read_csv(f"{BASE_URL}{files['orders']}")
    order_items = pd.read_csv(f"{BASE_URL}{files['order_items']}")
    products = pd.read_csv(f"{BASE_URL}{files['products']}")
    reviews = pd.read_csv(f"{BASE_URL}{files['reviews']}")

    print("\nDatasets successfully loaded into Memory:")
    print(f"- Orders: {orders.shape[0]} rows, {orders.shape[1]} columns.")
    print(f"- Order Items: {order_items.shape[0]} rows, {order_items.shape[1]} columns.")
    print(f"- Products: {products.shape[0]} rows, {products.shape[1]} columns.")
    print(f"- Reviews: {reviews.shape[0]} rows, {reviews.shape[1]} columns.")
except FileNotFoundError as e:
    print(f"Error loading from specified Google Drive path: {e}")
    print("Attempting to load from a generic local Colab content directory (/content/)...")
    # New fallback: try a different local path, e.g., /content/
    LOCAL_COLAB_PATH = "/content/"
    orders = pd.read_csv(f"{LOCAL_COLAB_PATH}{files['orders']}")
    order_items = pd.read_csv(f"{LOCAL_COLAB_PATH}{files['order_items']}")
    products = pd.read_csv(f"{LOCAL_COLAB_PATH}{files['products']}")
    reviews = pd.read_csv(f"{LOCAL_COLAB_PATH}{files['reviews']}")
    print("\nFallback Datasets successfully loaded from /content/:")
    print(f"- Orders: {orders.shape[0]} rows, {orders.shape[1]} columns.")
    print(f"- Order Items: {order_items.shape[0]} rows, {order_items.shape[1]} columns.")
    print(f"- Products: {products.shape[0]} rows, {products.shape[1]} columns.")
    print(f"- Reviews: {reviews.shape[0]} rows, {reviews.shape[1]} columns.")
except Exception as e:
    print(f"An unexpected error occurred during data loading: {e}")
    print("Please ensure the dataset files are correctly placed in either the specified Google Drive path or /content/.")

# SQL & Databases: From Data to Machine Learning

Welcome! In this class, we will explore how data transitions from raw transactions stored in a database to actionable predictions generated by Machine Learning models.

We will navigate through this complete modern workflow:
$$\text{Database} \longrightarrow \text{SQL} \longrightarrow \text{Data Extraction} \longrightarrow \text{Python (Pandas)} \longrightarrow \text{Data Preparation} \longrightarrow \text{Machine Learning}$$

### Core Concepts
* **What is Data?** Raw facts, figures, dates, and names. On its own, raw data lacks context.
* **What is a Database?** An organized collection of structured data stored electronically for easy access, management, and modification.
* **What is a Relational Database?** A database that organizes data into tables (rows and columns) that are linked—or *related*—to each other based on shared data keys.
* **What is SQL?** Structured Query Language (SQL) is the standardized programming language used to communicate with, manage, and extract data from relational databases.
* **What is a DBMS?** A Database Management System (DBMS) is software that interacts with users, applications, and the database itself to capture and analyze data.
* **SQL vs. Database System:** SQL is the *language* (the set of instructions), whereas SQLite, MySQL, and PostgreSQL are the *database systems* (the engines that execute those instructions and manage storage).

# Database Systems

### 1. Relational Database Engines
* **SQLite:** A lightweight, serverless database engine that stores data in a single file on your disk. It is ideal for local testing, mobile applications, and learning environments because it requires zero configuration.
* **MySQL:** A highly popular open-source, server-client database system designed for web applications and production-scale operational databases.
* **PostgreSQL:** An advanced, enterprise-grade open-source relational database known for its robustness, reliability, and support for complex data types.

### 2. Ecosystem Tools & Platforms
* **XAMPP:** A local development package containing Apache (web server), MariaDB/MySQL (database), and interpreters like PHP. It enables developers to host databases locally on their personal computers.
* **phpMyAdmin:** A web-based graphical interface used to manage MySQL or MariaDB databases easily without writing shell scripts.
* **Supabase:** A modern cloud platform (Backend-as-a-Service) built around a PostgreSQL database that provides APIs, user authentication, and storage out of the box.

### Core Architecture Diagrams

#### Conceptual Database Flow
```
SQL (Language)
   │
   ▼
Database System (SQLite / MySQL / PostgreSQL)
   │
   ▼
Tables (Rows & Columns)
   │
   ▼
Queries (Subsets of Data)
```

#### Local Development Architecture (XAMPP)
```
[User Web Browser] ──> [phpMyAdmin (GUI)] ──> [XAMPP Package] ──> [MySQL/MariaDB Engine]
```

#### Cloud Backend Architecture (Supabase)
```
[Client Application] ──> [Supabase API Dashboard] ──> [PostgreSQL Database]
```

# SQLite Practical & Setup

To make this live class interactive, we will convert the pre-loaded Olist Pandas DataFrames directly into an in-memory SQLite database. This gives us full access to standard SQL tables inside our environment.

In [ ]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database connection
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Olist DataFrames into SQLite tables
orders.to_sql('orders', conn, index=False, if_exists='replace')
order_items.to_sql('order_items', conn, index=False, if_exists='replace')
products.to_sql('products', conn, index=False, if_exists='replace')
reviews.to_sql('reviews', conn, index=False, if_exists='replace')

print("SQLite In-Memory Database initialized.")
print("Tables available: 'orders', 'order_items', 'products', 'reviews'")

### Exploring SQL Syntax Foundations

Unlike MySQL or PostgreSQL, SQLite does not require a `CREATE DATABASE` command since the database is instantiated directly on initialization. Let's practice basic SQL commands starting with table creation and querying.

In [ ]:
# 1. Create a temporary training sandbox table
cursor.execute("""
CREATE TABLE IF NOT EXISTS store_branches (
    branch_id INTEGER PRIMARY KEY AUTOINCREMENT,
    city TEXT NOT NULL,
    active_status INTEGER DEFAULT 1
);
""")

# 2. Insert values
cursor.execute("INSERT INTO store_branches (city, active_status) VALUES ('Sao Paulo', 1);")
cursor.execute("INSERT INTO store_branches (city, active_status) VALUES ('Rio de Janeiro', 1);")
cursor.execute("INSERT INTO store_branches (city, active_status) VALUES ('Belo Horizonte', 0);")
conn.commit()

# 3. Read back from our sandbox table
query = "SELECT * FROM store_branches;"
pd.read_sql_query(query, conn)

,branch_id,city,active_status
0,1,Sao Paulo,1
1,3,Belo Horizonte,1
2,4,Sao Paulo,1
3,5,Rio de Janeiro,1
4,6,Belo Horizonte,0
5,7,Sao Paulo,1
6,8,Rio de Janeiro,1
7,9,Belo Horizonte,0


The above query creates a table `store_branches`, inserts three branch locations, and executes a basic `SELECT *` query to display all columns and rows.

In [ ]:
# 4. Demonstrate UPDATE and DELETE
cursor.execute("UPDATE store_branches SET active_status = 1 WHERE city = 'Belo Horizonte';")
cursor.execute("DELETE FROM store_branches WHERE city = 'Rio de Janeiro';")
conn.commit()

query = "SELECT city, active_status FROM store_branches;"
pd.read_sql_query(query, conn)

,city,active_status
0,Sao Paulo,1
1,Belo Horizonte,1
2,Sao Paulo,1
3,Belo Horizonte,1
4,Sao Paulo,1
5,Belo Horizonte,1


The query above dynamically updates the status of the Belo Horizonte branch to active (`1`) and removes the Rio de Janeiro branch from the database using a `WHERE` condition.

# Working With a Realistic OLTP Dataset

Our database contains tables modeled on real e-commerce transactions (OLTP - Online Transaction Processing):

* **`orders`**: Logs transactional parameters (order status, timestamps, customer mappings).
* **`order_items`**: Logs product prices, item indices, shipping details, and seller records.
* **`products`**: Features descriptions of physical goods sold (categories, weight, dimensions).
* **`reviews`**: Logs post-delivery customer satisfaction ratings and feedback timestamps.

### Visualizing Database Relationships

```
    [orders]                        [order_items]
    - order_id  <─────────────────  - order_id
    - customer_id                   - product_id  ──────┐
    - order_status                  - price             │
                                                        │
    [reviews]                                           ▼
    - review_id                                     [products]
    - order_id  ────────────────┐                   - product_id
                                │                   - product_category_name
                                └────────────────>  
```

# Intermediate and Advanced SQL

Let's write intermediate SQL queries against our Olist tables to solve realistic operational questions.

In [ ]:
query = """
SELECT
    order_status,
    COUNT(order_id) AS total_orders
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC;
"""
pd.read_sql_query(query, conn)

,order_status,total_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


### Query Explanation
* **What it returns:** The total volume of orders segmented by their operational lifecycle statuses (e.g., delivered, shipped, canceled).
* **Why it matters:** Provides operations teams with a clear overview of bottlenecks, returns, or pending items.

In [ ]:
query = """
SELECT
    p.product_category_name,
    COUNT(oi.order_id) AS quantity_sold,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi INNER JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_category_name
HAVING total_revenue > 100000
ORDER BY total_revenue DESC;
"""
pd.read_sql_query(query, conn)

,product_category_name,quantity_sold,total_revenue
0,beleza_saude,9670,1258681.34
1,relogios_presentes,5991,1205005.68
2,cama_mesa_banho,11115,1036988.68
3,esporte_lazer,8641,988048.97
4,informatica_acessorios,7827,911954.32
5,moveis_decoracao,8334,729762.49
6,cool_stuff,3796,635290.85
7,utilidades_domesticas,6964,632248.66
8,automotivo,4235,592720.11
9,ferramentas_jardim,4347,485256.46


### Query Explanation
* **What it returns:** Top-performing categories generating over $100,000 in gross merchandise sales.
* **Why it matters:** Businesses use this data to focus marketing campaigns and optimize inventory strategies.

In [ ]:
query = """
SELECT
    o.order_id,
    o.order_status,
    COUNT(oi.product_id) AS items_count,
    ROUND(SUM(oi.price), 2) AS total_order_value,
    CASE
        WHEN SUM(oi.price) >= 200 THEN 'Premium'
        WHEN SUM(oi.price) >= 50 THEN 'Standard'
        ELSE 'Budget'
    END AS value_segment
FROM orders o LEFT JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY o.order_id
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,order_id,order_status,items_count,total_order_value,value_segment
0,00010242fe8c5a6d1ba2dd792cb16214,delivered,1,58.90,Standard
1,00018f77f2f0320c557190d7a144bdd3,delivered,1,239.90,Premium
2,000229ec398224ef6ca0657da4fc703e,delivered,1,199.00,Standard
3,00024acbcdf0a6daa1e931b038114c75,delivered,1,12.99,Budget
4,00042b26cf59d7ce69dfabb4e55b4fd9,delivered,1,199.90,Standard
5,00048cc3ae777c65dbb7d2a0634bc1ea,delivered,1,21.90,Budget
6,00054e8431b9d7675808bcb819fb4a32,delivered,1,19.90,Budget
7,000576fe39319847cbb9d288c5617fa6,delivered,1,810.00,Premium
8,0005a1a1728c9d785b8e2b08b904576c,delivered,1,145.95,Standard
9,0005f50442cb953dcd1d21e1fb923495,delivered,1,53.99,Standard


### Query Explanation
* **What it returns:** An order summary categorizing purchases into 'Premium', 'Standard', or 'Budget' based on purchase value.
* **Why it matters:** Helps build cohorts for targeted, customized marketing campaigns.
* *Note:* Right outer joins (`RIGHT JOIN`) are not native to default legacy engines like SQLite; `LEFT JOIN` structures are used instead to keep queries universally compatible.

# Assignment Questions
1. Find the distinct list of status updates in the `orders` table.
2. Find the minimum, average, and maximum price of items sold in the `order_items` table.
3. Count how many orders have been marked as `delivered`.
4. Write a join to return the product ID, category name, and review score for items that received a review score of `1`.
5. Find all product categories with an average purchase price above $150.

# SQL → Python

In standard production workflows, database systems act as reliable backends while Python libraries like Pandas analyze, transform, and visualize the retrieved tables.

```
Database (Stores raw logs) ────> SQL (Filters & extracts) ────> Pandas (Prepares data) ────> Machine Learning
```

Let's extract an aggregative customer subset directly into a Pandas DataFrame.

In [ ]:
query = """
SELECT
    order_id,
    price,
    freight_value
FROM order_items;
"""

df_items = pd.read_sql_query(query, conn)
print("Loaded into Pandas DataFrame shape:", df_items.shape)
display(df_items.head())

Loaded into Pandas DataFrame shape: (112650, 3)


,order_id,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14


### Comparing Syntax: SQL vs. Pandas

In [ ]:
# Goal: Filter transactions where product price > 200

# --- SQL Implementation ---
sql_filtered = pd.read_sql_query("SELECT * FROM order_items WHERE price > 200;", conn)

# --- Pandas Implementation ---
pandas_filtered = df_items[df_items['price'] > 200]

print(f"SQL Rows: {len(sql_filtered)} | Pandas Rows: {len(pandas_filtered)}")

SQL Rows: 13374 | Pandas Rows: 13374


# Moving From SQLite to XAMPP/MySQL

While SQLite is perfect for local notebooks, production scale architectures use dedicated multi-tenant systems.


## BACKUP LESSON: SQLite-Only Complete SQL Lab

*This is a fully self-contained database laboratory using the SQLite engine built directly into this Google Colab environment. No installations or external configurations are required.*

### 1. DDL & DML: Creating, Inserting, Updating & Deleting (Sandbox Table)

We start by using **Data Definition Language (DDL)** to define database structures, followed by **Data Manipulation Language (DML)** to modify data. In SQLite, the `CREATE DATABASE` statement is not required because SQLite automatically creates a file-based or memory-based database when establishing a connection.

In [ ]:
# --- CREATE TABLE & INSERT ---
cursor.execute("""
CREATE TABLE IF NOT EXISTS backup_branches (
    branch_id INTEGER PRIMARY KEY AUTOINCREMENT,
    branch_name TEXT NOT NULL,
    city TEXT NOT NULL,
    revenue REAL DEFAULT 0.00
);
""")

cursor.execute("INSERT INTO backup_branches (branch_name, city, revenue) VALUES ('South Hub', 'Sao Paulo', 150000.50);")
cursor.execute("INSERT INTO backup_branches (branch_name, city, revenue) VALUES ('North Hub', 'Rio de Janeiro', 85000.00);")
cursor.execute("INSERT INTO backup_branches (branch_name, city, revenue) VALUES ('West Hub', 'Belo Horizonte', 92000.75);")
conn.commit()

# Display newly created rows
pd.read_sql_query("SELECT * FROM backup_branches;", conn)

In [ ]:
# --- UPDATE & DELETE ---
# Updating the revenue of West Hub
cursor.execute("UPDATE backup_branches SET revenue = 98000.00 WHERE branch_name = 'West Hub';")

# Deleting North Hub
cursor.execute("DELETE FROM backup_branches WHERE branch_name = 'North Hub';")
conn.commit()

# Display final state
pd.read_sql_query("SELECT * FROM backup_branches;", conn)

### 2. SQL Basics & Core Queries
Now, let's explore retrieving and filtering data from the primary tables (`orders`, `order_items`, `products`, `reviews`).

In [ ]:
# --- SELECT, WHERE, ORDER BY, LIMIT, DISTINCT ---
query = """
SELECT DISTINCT
    order_status
FROM orders
WHERE order_status != 'delivered'
ORDER BY order_status ASC
LIMIT 5;
"""
pd.read_sql_query(query, conn)

,order_status
0,approved
1,canceled
2,created
3,invoiced
4,processing


### 3. Aggregations (COUNT, SUM, AVG, GROUP BY, HAVING)
Aggregations allow us to summarize records. We use `GROUP BY` to arrange data by categories and `HAVING` to filter groups (similar to how `WHERE` filters individual rows).

In [ ]:
query = """
SELECT
    product_id,
    COUNT(order_item_id) AS total_items_sold,
    SUM(price) AS total_sales_revenue,
    ROUND(AVG(price), 2) AS average_item_price
FROM order_items
GROUP BY product_id
HAVING total_items_sold > 50
ORDER BY total_sales_revenue DESC;
"""
pd.read_sql_query(query, conn)

,product_id,total_items_sold,total_sales_revenue,average_item_price
0,bb50f2e236e5eea0100680137654686c,195,63885.00,327.62
1,6cdd53843498f92890544667809f1595,156,54730.20,350.83
2,d1c427060a0f73f6b889a5c7c61f2ac4,343,47214.51,137.65
3,99a4788cb24856965c36a24e339b6058,488,43025.56,88.17
4,3dd2a17168ec895c781a9191c1e95ad7,274,41082.60,149.94
...,...,...,...,...
165,5a6e53c3b4e8684b13388d6aa4afdf12,73,1190.27,16.31
166,0152f69b6cf919bcdaf117aa8c43e5a2,64,981.60,15.34
167,ab1f9387c0627dd24000bfbc54fdeee0,69,890.14,12.90
168,4ae634441e444ca4bc85903cafe98d73,52,804.86,15.48


### 4. Advanced Operations: JOINs, Subqueries, CASE WHEN, and NULL Handling

#### A note on RIGHT JOIN in SQLite:
Unlike MySQL and PostgreSQL, legacy versions of SQLite do **NOT** support `RIGHT JOIN` or `FULL OUTER JOIN`. To perform a right outer join safely in SQLite without crashing, we must rewrite the tables sequence and use a `LEFT JOIN` instead.

* **Right Join logic:** `Table_A RIGHT JOIN Table_B` is functionally equivalent to `Table_B LEFT JOIN Table_A`.

In [ ]:
# --- INNER JOIN, LEFT JOIN, CASE WHEN, Aliases, and NULL Handling ---
query = """
SELECT
    o.order_id,
    o.order_status,
    COALESCE(r.review_score, 0) AS final_review_score,
    CASE
        WHEN r.review_score IS NULL THEN 'No Review'
        WHEN r.review_score >= 4 THEN 'Satisfied'
        ELSE 'Unsatisfied'
    END AS customer_sentiment
FROM orders o LEFT JOIN reviews r ON o.order_id = r.order_id
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,order_id,order_status,final_review_score,customer_sentiment
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,4,Satisfied
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,4,Satisfied
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,5,Satisfied
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,5,Satisfied
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,5,Satisfied
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,4,Satisfied
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,2,Unsatisfied
7,6514b8ad8028c9f2cc2374ded245783f,delivered,5,Satisfied
8,76c6e866289321a7c93b82b54852dc33,delivered,1,Unsatisfied
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,5,Satisfied


#### Multiple-Table JOINs & Subqueries
Let's calculate average performance metrics utilizing a multi-table hierarchy combined with nested logical queries.

# Assignment Continuation

In [ ]:
# Exercise 1: Find the average freight cost ('freight_value') in 'order_items'.


In [ ]:
# Exercise 2: List the top 5 highest-priced items in 'order_items' (display 'order_id', 'price', ordered high to low).


In [ ]:
# Exercise 3: Join 'order_items' and 'products' to get the category name for 'order_id' = '00010242fe8c5a6d1ba2dd792cb16214'.


In [ ]:
# Exercise 4: Show the distribution of reviews ('review_score' and count) grouped by score, where score is NOT NULL.


In [ ]:
# Exercise 5: Find the maximum price of a product sold in the 'cool_stuff' category.


# Machine Learning: From Our Olist Data to Predictions

Historical analysis tells us what happened. Machine Learning helps us predict what *will* happen based on trends.

* **Features ($X$):** Attributes we can observe (e.g., product weight, size, categories).
* **Target ($y$):** The outcome we want to predict (e.g., product price).
* **Classification:** Predicting discrete labels (e.g., High-Value vs. Low-Value).
* **Regression:** Predicting a continuous numeric value (e.g., forecasting delivery times or prices).

# scikit-learn Preview

Let's train a simple model to predict product price using properties from our Olist dataset.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

# 1. Merge tables to combine features and targets
ml_df = pd.merge(
    order_items[['product_id', 'price']],
    products[['product_id', 'product_weight_g', 'product_length_cm']],
    on='product_id'
).dropna()

# 2. Select Features (X) and Target (y)
X = ml_df[['product_weight_g', 'product_length_cm']]
y = ml_df['price']

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Choose & Train the Model
model = LinearRegression()
model.fit(X_train, y_train)



LinearRegression()

In [ ]:
# 5. Make Predictions
predictions = model.predict(X_test)



In [ ]:
# 6. Evaluate accuracy
rmse = root_mean_squared_error(y_test, predictions)
print(f"Model Trained successfully!")
print(f"Root Mean Squared Error: ${rmse:.2f}")

Model Trained successfully!
Root Mean Squared Error: $167.52


**Next: Building our first machine-learning model with scikit-learn.**